# 02 CHGNet, Mace(small, large)による計算<BR>
CHGNet, Mace によるパフォーマンスチェック（CPU/GPU 速度のみ）<BR>
様々なNNPの評価結果　https://matbench-discovery.materialsproject.org/ <BR>


作成：中山将伸 (2026.2.21)  再配布禁止<BR>

## 【必須】インストール　＋　ASEモジュール他のインポート


In [ ]:
!pip install ase
!pip install nglview
!pip install nglview ipywidgets==7.7.2
!pip install chgnet mace-torch

In [ ]:
import numpy as np
import math, random

import os,sys,csv,glob,shutil,re,time
from time import perf_counter
from joblib import Parallel, delayed
args = sys.argv

# sklearn
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt

import ase #
from ase.optimize import LBFGS, BFGS, FIRE
from ase.constraints import FixAtoms, FixedPlane, FixBondLength
from ase.filters import UnitCellFilter, ExpCellFilter
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary
from ase.md.verlet import VelocityVerlet
from ase.md.langevin import Langevin
from ase.md import MDLogger
from ase import Atoms
from ase.io import read, write
from ase.io import Trajectory
from ase import units
from ase.build import bulk, make_supercell
from ase.visualize import view

from google.colab import output
output.enable_custom_widget_manager()
import nglview as nv


## 【必須】Atoms object 入力

In [ ]:
#POSCAR ファイルの読み込み
!wget https://ndownloader.figshare.com/files/62065990
!mv 62065990 Li7La3Zr2O12.cif
atoms=read("./Li7La3Zr2O12.cif",format="cif")

## 【必須】Atoms objectにpretrained-機械学習ポテンシャルを導入

In [ ]:
import chgnet
import torch
from chgnet.model.dynamics import CHGNetCalculator

calc = CHGNetCalculator(
    use_device="cpu",          # "cuda" も可
    on_isolated_atoms="ignore" # よく使われる設定例
)

atoms.calc = calc

s_time = perf_counter()
energy=atoms.get_potential_energy()
force=atoms.get_forces()
e_time = perf_counter()

elapsedt=e_time-s_time
print ("computation time (chgnet_cpu)",elapsedt, "sec")




In [ ]:
from mace.calculators import mace_mp
import torch

# 軽量（small）
calc_mace_light = mace_mp(
    model="small",       # ←軽量
    device="cpu",        # "cuda" も可
    default_dtype="float32",
)

# ヘビー（large）
calc_mace_heavy = mace_mp(
    model="large",       # ←ヘビー
    device="cpu",
    default_dtype="float32",
)

# 使う方を差し替えるだけ
atoms.calc = calc_mace_light

s_time = perf_counter()
energy=atoms.get_potential_energy()
force=atoms.get_forces()
e_time = perf_counter()
elapsedt=e_time-s_time
print ("computation time(mace-light_cpu)",elapsedt, "sec")

atoms.calc = calc_mace_heavy
s_time = perf_counter()
energy=atoms.get_potential_energy()
force=atoms.get_forces()
e_time = perf_counter()
elapsedt=e_time-s_time
print ("computation time(mace-large_cpu)",elapsedt, "sec")

## GPUを使った計算

In [ ]:


calc = CHGNetCalculator(
    use_device="cuda",          # "cuda" も可
    on_isolated_atoms="ignore" # よく使われる設定例
)

atoms.calc = calc

torch.cuda.synchronize()
s_time = perf_counter()
energy=atoms.get_potential_energy()
force=atoms.get_forces()
torch.cuda.synchronize()
e_time = perf_counter()

elapsedt=e_time-s_time
print ("computation time (chgnet_cpu)",elapsedt, "sec")



# 軽量（small）
calc_mace_light = mace_mp(
    model="small",       # ←軽量
    device="cuda",        # "cuda" も可
    default_dtype="float32",
)

# ヘビー（large）
calc_mace_heavy = mace_mp(
    model="large",       # ←ヘビー
    device="cuda",
    default_dtype="float32",
)


atoms.calc = calc_mace_light

torch.cuda.synchronize()
s_time = perf_counter()
energy=atoms.get_potential_energy()
force=atoms.get_forces()
torch.cuda.synchronize()
e_time = perf_counter()
elapsedt=e_time-s_time
print ("computation time(mace-light_cpu)",elapsedt, "sec")

atoms.calc = calc_mace_heavy
torch.cuda.synchronize()
s_time = perf_counter()
energy=atoms.get_potential_energy()
force=atoms.get_forces()
torch.cuda.synchronize()
e_time = perf_counter()
elapsedt=e_time-s_time
print ("computation time(mace-large_cpu)",elapsedt, "sec")


In [ ]:
from mace.calculators import mace_mp

# 軽量（small）
calc_mace_light = mace_mp(
    model="small",       # ←軽量
    device="cpu",        # "cuda" も可
    default_dtype="float32",
)

# ヘビー（large）
calc_mace_heavy = mace_mp(
    model="large",       # ←ヘビー
    device="cpu",
    default_dtype="float32",
)

# 使う方を差し替えるだけ
atoms.calc = calc_mace_light

torch.cuda.synchronize()
s_time = perf_counter()
energy=atoms.get_potential_energy()
force=atoms.get_forces()
torch.cuda.synchronize()
e_time = perf_counter()
elapsedt=e_time-s_time
print ("computation time(mace-light_cpu)",elapsedt, "sec")

atoms.calc = calc_mace_heavy
torch.cuda.synchronize()
s_time = perf_counter()
energy=atoms.get_potential_energy()
force=atoms.get_forces()
torch.cuda.synchronize()
e_time = perf_counter()
elapsedt=e_time-s_time
print ("computation time(mace-large_cpu)",elapsedt, "sec")

----------------------------------------------------------------
# 03 ASEの使い方
MgO doped NiO と CaO doped NiOの固溶体形成

## Atoms object
結晶構造ファイルの読み込みは、 atoms.io.read でやる。<BR>
対応フォーマットは、　https://wiki.fysik.dtu.dk/ase/ase/io/io.html　　を参照<BR>


In [ ]:
# POSCARファイルの読み込み(ASE Atoms形式)  example NaCl
!wget https://ndownloader.figshare.com/files/62068150
!mv 62068150 NaCl.cif

inpf="./NaCl.cif"
atoms=read(inpf,format="cif")

print(atoms)

atoms_save=atoms.copy()  #元のatomsデータを保存  atoms_save=atoms  では、どちらか一方の結果が代わると、もう一方も変化してしまう

#### atoms objectからの各種情報の取り出し
  参考サイト   https://wiki.fysik.dtu.dk/ase/ase/atoms.html

In [ ]:

print(atoms.get_chemical_symbols())  # 元素の抽出（インデックス番号順）
print(atoms.positions)               # 原子の位置情報（ Cartesian 座標形式 )
print(type(atoms.positions))
print()
print(atoms.get_scaled_positions())  # 原子の位置情報（ Fractional(分率) 座標形式 )
print(type(atoms.get_scaled_positions()))


### 値の書き換えできるパラメーターと、参照しかできないパラメーターについて

In [ ]:
print ("以下のパラメーターは、代入することで修正可能")
print ("atoms.cell------------------------------")
print (atoms.cell)
print ("atoms.positions-------------------------　分率座標表記ではない")
print (atoms.positions)
print ("atoms.symbols---------------------------")
print (atoms.symbols[0],atoms.symbols[5],atoms.symbols[:])
print ("----------------------------------------")

print ("")
print ("以下のパラメーター引用は、代入不可（値を書き換えられない）")
print ("atoms.cell.cellpar()",atoms.cell.cellpar())
print ("atoms.get_number_of_atoms() ",atoms.get_number_of_atoms() )
print ("atoms.get_scaled_positions()",atoms.get_scaled_positions())
print ("atoms.get_chemical_formula('hill')",atoms.get_chemical_formula('hill'))
print ("atoms.get_chemical_symbols()",atoms.get_chemical_symbols() )


### リスト形式データの引用
引用した結果に、複数の数値が含まれている場合は、 [数字]で、特定のデータを引用できる

In [ ]:
print("------------------------------------------------------1")
print(atoms.positions)
print("------------------------------------------------------2")
print(atoms.positions[1])
print("------------------------------------------------------3")
print(atoms.positions[1][2])
print(atoms.positions[1,2])
print("------------------------------------------------------4")
print(atoms.positions[1][1:3])  #[1:3]は2列目から3列目を取得
print(atoms.positions[1,1:3])  #[1:3]は2列目から3列目を取得
print("------------------------------------------------------5")
print(atoms.positions[:][2])  #[:] は全データに対応
print(atoms.positions[:,2])  #[:] は全データに対応

In [ ]:
#結合長解析
distall=atoms.get_all_distances()      #全結合ペアーの結合長テーブル (原子数×原子数)
print("shape",distall.shape)
print(distall)
print("distance between index 0 and 1: {:1.6f} ang.".format(distall[0][1]))

In [ ]:
# Naイオンだけの情報を抽出したい (Naでかつ z分率座標が 0.4以上)
for i in range(len(atoms)):  # lenは対象となるリストやオブジェクトの数を抽出する関数（atomsの場合は原子数-1となる)
    if atoms.get_chemical_symbols()[i] == "Na" and atoms.get_scaled_positions()[i][2] >= 0.4:
        print ("Na", atoms.get_scaled_positions()[i])


## Trajectory データの取り扱い  (分子動力学法等で得られる軌跡ファイル)

In [ ]:
# 分子動力学法の軌跡データなど（MD 1step 毎のデータ )  Li112P16S96の場合

!wget https://ndownloader.figshare.com/files/62068279
!mv 62068279 MD_Li7PS6.traj
traj=read("./MD_Li7PS6.traj", index="1000:90000:10")
            # index=":"は全ステップデータ   "100:" 100step以降   ":100"  100stepまで
            #       "100:10000:10"  100ステップ目から10000ステップまで、10step毎に読み込み
#traj=read("./inputs/XDATCAR", index=":", format="vasp-xdatcar")  # index=":"は全ステップデータ   "100:" 100step以降   ":100"  100stepまで

print(len(traj))  #trajの長さ＝MDステップ数
print(len(traj[-1]))  #traj[-1] 最後のステップの記録  = 構造中に含まれる現数数
print(traj[0].get_chemical_formula())


### Atoms object  (ase.visualize)

!pip install nglview --upgrade<BR>
jupyter notebookをanacondaから立ち上げる際に、<BR>
$jupyter notebook --NotebookApp.iopub_data_rate_limit=10000000 <BR>
としないと動作しないことがある。(jupyter versionによる？）


In [ ]:
# google colabではなぜか動かない。
view(traj, viewer="ngl")


In [ ]:
view(traj[800], viewer="ngl")

In [ ]:
# インデックス番号0 と 1のイオン間距離を時間tに対してモニター

dist=traj[0].get_distance(0,1, mic=True)
print ("distance at t=0:",dist)
print ("")

#---
dist01=[]
for t in range(len(traj)):
    print (t, traj[t].get_distance(0,1, mic=True))
    dist01.append(traj[t].get_distance(0,1, mic=True))

plt.plot(dist01)


## Atoms object (ase.build)

atomsを加工してモデルを作成<BR>

In [ ]:
# 超構造作成
from ase.build import make_supercell
from ase.build import sort as asesort

P=[[3,0,0],[0,3,0],[0,0,3]]
atoms_sc=make_supercell(atoms,P)  # order キーワードで 元素の順番を変えられるはずだが動作しない。
cs=atoms_sc.get_chemical_symbols()        #構成原子の元素　(1 x n 行列)
print(cs)
print("---------------------------------------------------------------------------------------------------")
atoms_sorted=asesort(atoms_sc)  # 代わりにsortで並び替え

ase.io.write("supercell.cif",atoms_sorted,format="cif")

cs=atoms_sorted.get_chemical_symbols()        #構成原子の元素　(1 x n 行列)
print(cs)

atoms=atoms_sorted.copy()

In [ ]:
from ase.visualize import view as aseview
aseview(atoms,viewer='ngl')


## Atoms object (ase.geometry.analysis)

結晶構造データの解析（advanced）<BR>
参考サイト　https://wiki.fysik.dtu.dk/ase/ase/geometry.html

In [ ]:
atoms=atoms_save.copy()
atoms

In [ ]:
# RDF計算
r=[]
s_rdf=[]

from ase.geometry.analysis import Analysis
analysis=Analysis(atoms_sorted)  #インスタンス化

for i in range(50):
    r.append(i/50*8)

rdf=analysis.get_rdf(8,50,elements=('Na','Cl'))
s_rdf.append(rdf[0])
rdf=analysis.get_rdf(8,50,elements=('Na','Na'))
s_rdf.append(rdf[0])

plt.plot(r,s_rdf[0])
plt.plot(r,s_rdf[1])

#RDFのファイル出力 Na-Clの場合
f=open('rdf_Na-Cl.dat','w')
for i in range(50):
    print (r[i], s_rdf[1][i],file=f)
f.close()



## Atomsデータの書き換え

In [ ]:
atomscopy=atoms.copy()
#atoms_new=atoms_sorted.copy()
#注意  atomscopy = atoms だと、今後atomscopyを修正すると、atomsも修正されてしまう。

### 任意原子の座標修正

In [ ]:
#分率座標変換
scaled_positions = atomscopy.get_scaled_positions()
print(scaled_positions)

scaled_positions[0]=[0.2,0.4,0.1]  # 1番目の元素の位置を強制的に変更

#カーテシアン座標に再変換

cart_positions=[]
for i in range(len(atomscopy)):
    cart_positions.append(list(np.dot(atomscopy.cell,scaled_positions[i])))

print(cart_positions)

atomscopy.positions=cart_positions

### 任意原子の元素置換 Na-->K


In [ ]:
print(atomscopy.symbols)
atomscopy.symbols[198]='K'
print(atomscopy.symbols)

In [ ]:
# ファイルの生成
write("kNa107Cl108.cif",atomscopy,format="cif")

### 任意原子の消去（空孔生成）

In [ ]:
del atomscopy[0]  #0番目の原子情報を消去（空孔）　この場合はCl原子

# ファイルの生成
write("kNa107Cl107.cif",atomscopy,format="cif")

### 乱数を使って元素置換

In [ ]:
import random

atomscopy=atoms.copy()

listNa=[]
for i in range(len(atomscopy)):
    if atomscopy[i].symbol=='Na':
        listNa.append(i)

# ランダムに4つのNaを選び、該当するindex番号を random_numbersに格納する。
random_numbers = random.sample(listNa, 4)
print(random_numbers)

for ri in random_numbers:
    atomscopy.symbols[ri]='K'

print(atomscopy.symbols)
print(atomscopy.get_chemical_formula('hill'))

write("K_doped.cif",atomscopy,format="cif")


# 練習問題

岩塩型NiO中にMgOとCaOをそれぞれ微量ドープさせた場合に雇用するかどうか評価するスクリプトを作る。

In [ ]:
atoms=atoms_save.copy()
atoms.symbols='Ni4O4'  #NaClをNiOに読み替える

# 超構造作成
from ase.build import make_supercell
from ase.build import sort as asesort

P=[[2,0,0],[0,2,0],[0,0,2]]
atoms_sc=make_supercell(atoms,P)  # order キーワードで 元素の順番を変えられるはずだが動作しない。
cs=atoms_sc.get_chemical_symbols()        #構成原子の元素　(1 x n 行列)
atoms_sorted=asesort(atoms_sc)  # 代わりにsortで並び替え
ase.io.write("supercell.cif",atoms_sorted,format="cif")

cs=atoms_sorted.get_chemical_symbols()        #構成原子の元素　(1 x n 行列)
atoms=atoms_sorted.copy()
atoms

In [ ]:
calc = CHGNetCalculator(
    use_device="cpu",          # "cuda" も可
    on_isolated_atoms="ignore" # よく使われる設定例
)

atoms.symbols='Ni32O32'
atoms.calc = calc
ucf=ExpCellFilter(atoms)
opt = LBFGS(ucf, trajectory="relax.traj",logfile="log.relax", maxstep=0.02)
opt.run(steps=100)
energy_NiO=atoms.get_potential_energy()
print("E(Ni32O32):",energy_NiO)

atoms.symbols='Mg32O32'
atoms.calc = calc
ucf=ExpCellFilter(atoms)
opt = LBFGS(ucf, trajectory="relax.traj",logfile="log.relax", maxstep=0.02)
opt.run(steps=100)
energy_MgO=atoms.get_potential_energy()
print("E(Mg32O32)",energy_MgO)

atoms.symbols='Ca32O32'
atoms.calc = calc
ucf=ExpCellFilter(atoms)
opt = LBFGS(ucf, trajectory="relax.traj",logfile="log.relax", maxstep=0.02)
opt.run(steps=100)
energy_CaO=atoms.get_potential_energy()
print("E(Ca32O32)",energy_CaO)

atoms.symbols='MgNi31O32'
atoms.calc = calc
ucf=ExpCellFilter(atoms)
opt = LBFGS(ucf, trajectory="relax.traj",logfile="log.relax", maxstep=0.02)
opt.run(steps=100)
energy_MgNiO=atoms.get_potential_energy()
print("E(Mg1Ni31O32)",energy_MgNiO)

atoms.symbols='CaNi31O32'
atoms.calc = calc
ucf=ExpCellFilter(atoms)
opt = LBFGS(ucf, trajectory="relax.traj",logfile="log.relax", maxstep=0.02)
opt.run(steps=100)
energy_CaNiO=atoms.get_potential_energy()
print("E(Ca1Ni31O32)",energy_CaNiO)


